In [74]:
import pandas as pd
test_df_eng = pd.read_parquet('datasets/test_df_eng.parquet', engine='fastparquet')

In [75]:
base_features = [
    'Hour', 'Day_of_Week', 'Month', 'Weekend', 'Holiday', 'Year', 'Zone_Int_ID', 'Amenity', 'Crossing', 'Give_Way', 'Junction',
    'Railway', 'Station', 'Stop', 'Traffic_Signal', 'City_Houston', 'City_Miami', 'Temperature(F)', 'Humidity(%)', 'Pressure(in)',
    'Visibility(mi)', 'Wind_Speed(mph)', 'Precipitation(in)', 'Weather_Clear', 'Weather_Cloudy', 'Weather_Dust/Windy',
    'Weather_Rain/Drizzle', 'Weather_Snow/Ice', 'Weather_Stormy', 'Weather_Visibility Issues'
]

engineered_features = [
    'Wind_x_Precip', 'Hour_x_Weekend', 'BadWeather', 'BadWeather_x_Hour', 'Hour_squared', 'WindSpeed_squared', 'Precip_squared',
    'Hour_sin', 'Hour_cos', 'Month_sin', 'Month_cos', 'DayOfWeek_sin', 'DayOfWeek_cos', 'Zone_Mean', 'Zone_Std', 'Zone_Max'
]

features = base_features + engineered_features
print(f"Total features: {len(features)}")

Total features: 46


In [76]:
test_df_eng[features]

,Hour,Day_of_Week,Month,Weekend,Holiday,Year,Zone_Int_ID,Amenity,Crossing,Give_Way,...,Precip_squared,Hour_sin,Hour_cos,Month_sin,Month_cos,DayOfWeek_sin,DayOfWeek_cos,Zone_Mean,Zone_Std,Zone_Max
0,10,2,3,0,0,2022,0,0.034011,0.188304,0.007051,...,0.0,0.500000,-0.866025,1.0,6.123234e-17,0.974928,-0.222521,0.567745,1.064073,9
1,10,2,3,0,0,2022,1,0.021739,0.231121,0.041190,...,0.0,0.500000,-0.866025,1.0,6.123234e-17,0.974928,-0.222521,0.176973,0.501770,7
2,10,2,3,0,0,2022,2,0.000000,0.133127,0.040248,...,0.0,0.500000,-0.866025,1.0,6.123234e-17,0.974928,-0.222521,0.083973,0.332614,6
3,10,2,3,0,0,2022,3,0.000000,0.062893,0.000000,...,0.0,0.500000,-0.866025,1.0,6.123234e-17,0.974928,-0.222521,0.038218,0.221691,4
4,10,2,3,0,0,2022,4,0.000000,0.015075,0.000000,...,0.0,0.500000,-0.866025,1.0,6.123234e-17,0.974928,-0.222521,0.057615,0.263937,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
705307,8,3,3,0,0,2023,153,0.000000,0.000000,0.000000,...,0.0,0.866025,-0.500000,1.0,6.123234e-17,0.433884,-0.900969,0.000000,0.000000,0
705308,8,3,3,0,0,2023,154,0.000000,0.000000,0.000000,...,0.0,0.866025,-0.500000,1.0,6.123234e-17,0.433884,-0.900969,0.000000,0.000000,0
705309,8,3,3,0,0,2023,155,0.000000,0.000000,0.000000,...,0.0,0.866025,-0.500000,1.0,6.123234e-17,0.433884,-0.900969,0.000000,0.000000,0
705310,8,3,3,0,0,2023,156,0.000000,0.000000,0.000000,...,0.0,0.866025,-0.500000,1.0,6.123234e-17,0.433884,-0.900969,0.000000,0.000000,0


In [77]:
import joblib

classifier = joblib.load('models/xgb_classifier_model.pkl')
regressor = joblib.load('models/xgb_regressor_model.pkl')
classifier2 = joblib.load('models/xgb_classifier_model2.pkl')
regressor2 = joblib.load('models/xgb_regressor_model2.pkl')


## Feature Importance

In [78]:
# Finding the top 10 most important features for the classifier model
classifier_importance = pd.DataFrame({'Feature': features, 'Importance': classifier.feature_importances_})
classifier_importance = classifier_importance.sort_values(by='Importance', ascending=False).head(10)

In [79]:
classifier_importance

,Feature,Importance
44,Zone_Std,0.523208
43,Zone_Mean,0.206938
38,Hour_cos,0.041687
1,Day_of_Week,0.028031
24,Weather_Cloudy,0.021174
5,Year,0.020450
2,Month,0.014283
20,Visibility(mi),0.013819
0,Hour,0.012384
40,Month_cos,0.011926


In [80]:
# Finding the top 10 most important features for the regressor model
regressor_importance = pd.DataFrame({'Feature': features, 'Importance': regressor.feature_importances_})
regressor_importance = regressor_importance.sort_values(by='Importance', ascending=False).head(10)

In [81]:
regressor_importance

,Feature,Importance
12,Station,0.080130
34,Hour_squared,0.072625
38,Hour_cos,0.061959
5,Year,0.049411
3,Weekend,0.041795
44,Zone_Std,0.041132
0,Hour,0.036759
45,Zone_Max,0.032672
2,Month,0.029784
1,Day_of_Week,0.028957


## Final Model Metrics on Test

In [82]:
X_test_classifier = pd.read_parquet('datasets/X_test_classifier.parquet', engine='fastparquet')

In [83]:
y_test_classifier = pd.read_csv('datasets/y_test_classifier.csv').squeeze('columns')

### Classifier

In [84]:
# Classifier predictions
y_test_probs = classifier.predict_proba(X_test_classifier)[:, 1]
y_test_preds = (y_test_probs >= 0.4).astype(int)

In [85]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score, precision_score

print("Classification Report:")
print(classification_report(y_test_classifier, y_test_preds))

print("Confusion Matrix:")
print(confusion_matrix(y_test_classifier, y_test_preds))

print(f"ROC AUC Score: {roc_auc_score(y_test_classifier, y_test_probs):.4f}")

print(f"Average Precision Score: {average_precision_score(y_test_classifier, y_test_probs)}")

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.74      0.85    670990
           1       0.15      0.88      0.26     34322

    accuracy                           0.75    705312
   macro avg       0.57      0.81      0.55    705312
weighted avg       0.95      0.75      0.82    705312

Confusion Matrix:
[[499707 171283]
 [  4266  30056]]
ROC AUC Score: 0.8933
Average Precision Score: 0.32547594583908646


In [86]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

# threshold sweep on test set (diagnostic only)
threshold_grid = np.arange(0.05, 0.96, 0.01)
rows = []

for threshold in threshold_grid:
    preds = (y_test_probs >= threshold).astype(int)
    p = precision_score(y_test_classifier, preds, zero_division=0)
    r = recall_score(y_test_classifier, preds, zero_division=0)
    f1 = f1_score(y_test_classifier, preds, zero_division=0)
    rows.append((threshold, p, r, f1))

threshold_results = pd.DataFrame(rows, columns=['threshold', 'precision', 'recall', 'f1'])
best_f1_row = threshold_results.loc[threshold_results['f1'].idxmax()]

print('Best threshold on this test sweep (diagnostic):')
print(best_f1_row)
print('\nTop 10 thresholds by F1:')
print(threshold_results.sort_values('f1', ascending=False).head(10))

Best threshold on this test sweep (diagnostic):
threshold    0.820000
precision    0.324137
recall       0.451518
f1           0.377368
Name: 77, dtype: float64

Top 10 thresholds by F1:
    threshold  precision    recall        f1
77       0.82   0.324137  0.451518  0.377368
78       0.83   0.334562  0.431123  0.376754
76       0.81   0.313121  0.471068  0.376188
75       0.80   0.303228  0.489016  0.374338
79       0.84   0.347137  0.406153  0.374334
74       0.79   0.294515  0.507838  0.372818
73       0.78   0.286043  0.524765  0.370261
80       0.85   0.360012  0.378620  0.369082
72       0.77   0.278044  0.540324  0.367155
71       0.76   0.271348  0.555416  0.364580


In [87]:
# compare saved classifier artifacts at the same threshold
for model_name, model in [
    ('models/xgb_classifier_model.pkl', classifier),
    ('xgb_classifier_model2.pkl', classifier2)
]:
    probs = model.predict_proba(X_test_classifier)[:, 1]
    preds = (probs >= 0.4).astype(int)
    print(f"\n{model_name}")
    print(f"ROC AUC: {roc_auc_score(y_test_classifier, probs):.4f}")
    print(f"PR AUC: {average_precision_score(y_test_classifier, probs):.4f}")
    print(f"Precision@0.4: {precision_score(y_test_classifier, preds, zero_division=0):.4f}")
    print(f"Recall@0.4: {recall_score(y_test_classifier, preds, zero_division=0):.4f}")
    print(f"F1@0.4: {f1_score(y_test_classifier, preds, zero_division=0):.4f}")


models/xgb_classifier_model.pkl
ROC AUC: 0.8933
PR AUC: 0.3255
Precision@0.4: 0.1493
Recall@0.4: 0.8757
F1@0.4: 0.2551

xgb_classifier_model2.pkl
ROC AUC: 0.8928
PR AUC: 0.3244
Precision@0.4: 0.1490
Recall@0.4: 0.8748
F1@0.4: 0.2546


### Regressor

In [88]:
import numpy as np
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from scipy.stats import spearmanr

# Evaluate regressor only on actual accident rows in test set
test_accidents = test_df_eng[test_df_eng['Is_Accident'] == 1].copy()
X_test_regressor = test_accidents[features]
y_test_regressor = test_accidents['Accident_Count']
y_test_log = np.log1p(y_test_regressor)

y_test_log_preds = regressor.predict(X_test_regressor)
y_test_final_preds = np.expm1(y_test_log_preds)

mae = mean_absolute_error(y_test_regressor, y_test_final_preds)
rmse = root_mean_squared_error(y_test_regressor, y_test_final_preds)
r2_raw = r2_score(y_test_regressor, y_test_final_preds)
r2_log = r2_score(y_test_log, y_test_log_preds)

# Spearman rank correlation
spearman_corr, spearman_pval = spearmanr(y_test_regressor, y_test_final_preds)

# Weighted Mean Absolute Error (weighted by actual severity values)
weights = y_test_regressor / y_test_regressor.sum() if y_test_regressor.sum() > 0 else np.ones_like(y_test_regressor) / len(y_test_regressor)
wmae = np.sum(weights * np.abs(y_test_regressor - y_test_final_preds))

print(f"Testing on {len(X_test_regressor)} accident rows")
print(f"MAE: {round(mae, 4)}")
print(f"RMSE: {round(rmse, 4)}")
print(f"R² (raw): {round(r2_raw, 4)}")
print(f"R² (log space): {round(r2_log, 4)}")
print(f"Spearman Rank Correlation: {round(spearman_corr, 4)} (p-value: {round(spearman_pval, 6)})")
print(f"WMAE: {round(wmae, 4)}")

Testing on 34322 accident rows
MAE: 1.3227
RMSE: 3.2189
R² (raw): 0.0412
R² (log space): -0.0646
Spearman Rank Correlation: 0.3336 (p-value: 0.0)
WMAE: 5.6871
